In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim

import numpy as np
from tqdm import tqdm

import random

import copy

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [2]:
import warnings
warnings.filterwarnings("ignore", category=ResourceWarning)

In [3]:
device

device(type='cuda')

In [4]:
def save_checkpoint(model, optimizer, epoch, filename="checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved to {filename}")

def load_checkpoint(model, optimizer, filename="checkpoint.pth"):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    print(f"Checkpoint loaded from {filename}, starting from epoch {epoch}")
    return epoch

# Модель - WRN

In [5]:
# -------------------------------
# WideResNet
# -------------------------------

# на основе https://github.com/bmsookim/wide-resnet.pytorch/

class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropout_rate=0.0):
        super(BasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != out_planes:
            self.shortcut = nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

    def forward(self, x):
        out = self.bn1(x)
        out = self.relu(out)
        out = self.conv1(out)
        out = self.dropout(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv2(out)
        out += self.shortcut(x)
        return out


class NetworkBlock(nn.Module):
    def __init__(self, num_layers, in_planes, out_planes, block, stride, dropout_rate):
        super(NetworkBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            s = stride if i == 0 else 1
            layers.append(block(in_planes if i == 0 else out_planes, out_planes, s, dropout_rate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=2, num_classes=10, dropout_rate=0.0):
        super(WideResNet, self).__init__()
        # проверим корректность глубины сети (должна быть 6n+4)
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6 # колво блоков в каждой группе
        k = widen_factor
        nStages = [16, 16 * k, 32 * k, 64 * k] # колво каналов на каждом этапе

        self.conv1 = nn.Conv2d(3, nStages[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nStages[0], nStages[1], BasicBlock, 1, dropout_rate)
        self.block2 = NetworkBlock(n, nStages[1], nStages[2], BasicBlock, 2, dropout_rate)
        self.block3 = NetworkBlock(n, nStages[2], nStages[3], BasicBlock, 2, dropout_rate)

        # Финальные слои
        self.bn1 = nn.BatchNorm2d(nStages[3])
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(nStages[3], num_classes)

        self._initialize_weights()

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out) #тут уменьшили размерность
        out = self.block3(out) #тут уменьшили размерность
        out = self.bn1(out)
        out = self.relu(out)
        out = F.adaptive_avg_pool2d(out, 1) # глобальное усреднение
        out = out.view(out.size(0), -1)   # развернули в вектор
        return self.fc(out)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1) # задаем веса batchnorm как 1
                nn.init.constant_(m.bias, 0)   # смещения bn как 0
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)


# грузим cifar10

In [6]:

# аугментация и предобработка
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),    #https://pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html
    transforms.RandomCrop(32, padding=4), #https://pytorch.org/vision/main/generated/torchvision.transforms.RandomCrop.html
    transforms.ToTensor(),                # (нормализация в [0, 1]) https://pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)) # нормализация по каналам
])

# датасет
trainset = torchvision.datasets.CIFAR10(root='./data_CIFAR10', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data_CIFAR10', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# модель, критерий, оптимизатор
model = WideResNet(depth=28, widen_factor=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)

100%|██████████| 170M/170M [00:02<00:00, 70.2MB/s]


# просто IMP

In [7]:
class Trainer:
    def __init__(self, model, trainloader, testloader, criterion, optimizer, scheduler=None):
        self.model = model
        self.trainloader = trainloader
        self.testloader = testloader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler

        self.train_losses = []
        self.test_losses = []
        self.train_accs = []
        self.test_accs = []

    def train(self, epoch):
        self.model.train()
        correct, total, running_loss = 0, 0, 0
        for inputs, targets in self.trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()

            # сбор статистики
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        acc = 100. * correct / total
        self.train_losses.append(running_loss / len(self.trainloader))
        self.train_accs.append(acc)
        print(f"[{epoch}] Train Loss: {self.train_losses[-1]:.3f} | Acc: {acc:.2f}%")

    def test(self):
        self.model.eval()
        correct, total, running_loss = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in self.testloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        acc = 100. * correct / total
        self.test_losses.append(running_loss / len(self.testloader))
        self.test_accs.append(acc)
        print(f"     Test Loss: {self.test_losses[-1]:.3f} | Acc: {acc:.2f}%")

    # сохраняется текущее состаяние оптимизатора и модели
    def save_checkpoint(self, epoch, filename="checkpoint.pth"):
        save_checkpoint(self.model, self.optimizer, epoch, filename)
    # загружаем их же
    def load_checkpoint(self, filename="checkpoint.pth"):
        epoch = load_checkpoint(self.model, self.optimizer, filename)
        return epoch

    def plot_metrics(self):
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_losses, label='Train Loss')
        plt.plot(self.test_losses, label='Test Loss')
        plt.title('Loss')
        plt.xlabel('Epoch')
        plt.grid()
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(self.train_accs, label='Train Acc')
        plt.plot(self.test_accs, label='Test Acc')
        plt.title('Accuracy')
        plt.xlabel('Epoch')
        plt.grid()
        plt.legend()

        plt.tight_layout()
        plt.show()


# 2 время инференса, колво ненулевых параметров,акураси на тесте

In [8]:

# аугментация и предобработка
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),    #https://pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html
    transforms.RandomCrop(32, padding=4), #https://pytorch.org/vision/main/generated/torchvision.transforms.RandomCrop.html
    transforms.ToTensor(),                # (нормализация в [0, 1]) https://pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)) # нормализация по каналам
])

# датасет
trainset = torchvision.datasets.CIFAR10(root='./data_CIFAR10', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data_CIFAR10', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# модель, критерий, оптимизатор
model = WideResNet(depth=28, widen_factor=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)

In [10]:
model_test_imp1_wrn = WideResNet(depth=28, widen_factor=2).to(device)
model_test_imp1_wrn.load_state_dict(torch.load("imp1_wrn28_2_s4.pth"))

model_test_swamp_wrn = WideResNet(depth=28, widen_factor=2).to(device)

model_test_swamp_wrn.load_state_dict(torch.load("swamp_wrn28_2_s7.pth"))

<All keys matched successfully>

# Sparsity

In [11]:
# Загрузка модели и маски
model_test_swamp_wrn = WideResNet(depth=28, widen_factor=2).to(device)
model_test_swamp_wrn.load_state_dict(torch.load("swamp_wrn28_2_s7.pth"))
swamp_mask = torch.load("swamp_mask.pt")

# Применение маски и подсчёт
total_params = 0
nonzero_params = 0

for name, param in model_test_swamp_wrn.named_parameters():
    if name in swamp_mask:
        mask = swamp_mask[name].to(device)
        param.data = param.data * mask
        nonzero_params += mask.sum().item()
        total_params += mask.numel()
    else:
        nonzero_params += (param != 0).sum().item()
        total_params += param.numel()

sparsity = 1 - (nonzero_params / total_params)
print(f"Total params: {total_params}")
print(f"Non-zero params: {nonzero_params}")
print(f"Sparsity: {sparsity:.2%}")

Total params: 1467610
Non-zero params: 309303
Sparsity: 78.92%


In [16]:
import time
import torch

In [12]:
# Загрузка модели и маски
model_test_imp_wrn = WideResNet(depth=28, widen_factor=2).to(device)
model_test_imp_wrn.load_state_dict(torch.load("imp1_wrn28_2_s4.pth"))
imp_mask = torch.load("imp1_mask.pt")

# Применение маски и подсчёт
total_params = 0
nonzero_params = 0

for name, param in model_test_imp_wrn.named_parameters():
    if name in imp_mask:
        mask = imp_mask[name].to(device)
        param.data = param.data * mask
        nonzero_params += mask.sum().item()
        total_params += mask.numel()
    else:
        nonzero_params += (param != 0).sum().item()
        total_params += param.numel()

sparsity = 1 - (nonzero_params / total_params)
print(f"Total params: {total_params}")
print(f"Non-zero params: {nonzero_params}")
print(f"Sparsity: {sparsity:.2%}")

Total params: 1467610
Non-zero params: 602276
Sparsity: 58.96%


# inference time

In [17]:
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)


In [18]:
model_test_imp_wrn.eval()
batch_size = testloader.batch_size

# Прогрев
dummy_input = torch.randn(batch_size, *testloader.dataset[0][0].shape).to(device)
for _ in range(10):
    _ = model_test_imp_wrn(dummy_input)
torch.cuda.synchronize()

# Основные замеры
num_runs = 5
total_time = 0

with torch.no_grad():
    for _ in range(num_runs):
        for data, _ in testloader:
            data = data.to(device)
            start = time.time()
            _ = model_test_imp_wrn(data)
            torch.cuda.synchronize()
            total_time += time.time() - start

    avg_time_per_sample = total_time / (num_runs * len(testloader.dataset))
    fps = (num_runs * len(testloader.dataset)) / total_time

print('IMP')
print(f"Average latency per sample: {avg_time_per_sample*1000:.2f} ms")
print(f"Throughput: {fps:.2f} samples/sec")

IMP
Average latency per sample: 0.17 ms
Throughput: 5891.93 samples/sec


In [19]:
model_test_swamp_wrn.eval()
batch_size = testloader.batch_size

# Прогрев
dummy_input = torch.randn(batch_size, *testloader.dataset[0][0].shape).to(device)
for _ in range(10):
    _ = model_test_swamp_wrn(dummy_input)
torch.cuda.synchronize()

# Основные замеры
num_runs = 5
total_time = 0

with torch.no_grad():
    for _ in range(num_runs):
        for data, _ in testloader:
            data = data.to(device)
            start = time.time()
            _ = model_test_swamp_wrn(data)
            torch.cuda.synchronize()
            total_time += time.time() - start

    avg_time_per_sample = total_time / (num_runs * len(testloader.dataset))
    fps = (num_runs * len(testloader.dataset)) / total_time

print('SWAMP')
print(f"Average latency per sample: {avg_time_per_sample*1000:.2f} ms")
print(f"Throughput: {fps:.2f} samples/sec")

SWAMP
Average latency per sample: 0.16 ms
Throughput: 6118.13 samples/sec


In [ ]:
import torch
import time

# Создаем тестовый входной тензор (предполагая, что вход - 3 канала, 32x32 как в CIFAR)
batch_size = 128
dummy_input = torch.randn(batch_size, 3, 32, 32).to(device)

# Функция для измерения времени inference
def measure_inference_time(model, input_tensor, num_runs=100):
    # warm-up
    for _ in range(10):
        _ = model(input_tensor)

    # Синхронизация перед началом измерений
    torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        _ = model(input_tensor)
    torch.cuda.synchronize()
    end_time = time.time()

    avg_time = (end_time - start_time) / num_runs
    return avg_time

# Измеряем время для swamp модели
model_test_swamp_wrn.eval()  # Переводим в режим оценки
with torch.no_grad():  # Отключаем вычисление градиентов
    swamp_time = measure_inference_time(model_test_swamp_wrn, dummy_input)
    print(f"Swamp WRN28-2 average inference time: {swamp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"Swamp WRN28-2 average inference time per sample: {swamp_time/batch_size:.6f} seconds")

# Измеряем время для imp модели
model_test_imp_wrn.eval()
with torch.no_grad():
    imp_time = measure_inference_time(model_test_imp_wrn, dummy_input)
    print(f"IMP WRN28-2 average inference time: {imp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"IMP WRN28-2 average inference time per sample: {imp_time/batch_size:.6f} seconds")

# Сравнение
speedup = imp_time / swamp_time if imp_time > swamp_time else swamp_time / imp_time
if imp_time < swamp_time:
    print(f"IMP модель быстрее на {speedup:.2f}x")
else:
    print(f"Swamp модель быстрее на {speedup:.2f}x")

Swamp WRN28-2 average inference time: 0.0147 seconds per batch (128 samples)
Swamp WRN28-2 average inference time per sample: 0.000115 seconds
IMP WRN28-2 average inference time: 0.0149 seconds per batch (128 samples)
IMP WRN28-2 average inference time per sample: 0.000116 seconds
Swamp модель быстрее на 1.01x


In [ ]:
import torch
import time

# Создаем тестовый входной тензор (предполагая, что вход - 3 канала, 32x32 как в CIFAR)
batch_size = 128
dummy_input = torch.randn(batch_size, 3, 32, 32).to(device)

# Функция для измерения времени inference
def measure_inference_time(model, input_tensor, num_runs=1000):
    # warm-up
    for _ in range(10):
        _ = model(input_tensor)

    # Синхронизация перед началом измерений
    torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        _ = model(input_tensor)
    torch.cuda.synchronize()
    end_time = time.time()

    avg_time = (end_time - start_time) / num_runs
    return avg_time

# Измеряем время для swamp модели
model_test_swamp_wrn.eval()  # Переводим в режим оценки
with torch.no_grad():  # Отключаем вычисление градиентов
    swamp_time = measure_inference_time(model_test_swamp_wrn, dummy_input)
    print(f"Swamp WRN28-2 average inference time: {swamp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"Swamp WRN28-2 average inference time per sample: {swamp_time/batch_size:.6f} seconds")

# Измеряем время для imp модели
model_test_imp_wrn.eval()
with torch.no_grad():
    imp_time = measure_inference_time(model_test_imp_wrn, dummy_input)
    print(f"IMP WRN28-2 average inference time: {imp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"IMP WRN28-2 average inference time per sample: {imp_time/batch_size:.6f} seconds")

# Сравнение
speedup = imp_time / swamp_time if imp_time > swamp_time else swamp_time / imp_time
if imp_time < swamp_time:
    print(f"IMP модель быстрее на {speedup:.2f}x")
else:
    print(f"Swamp модель быстрее на {speedup:.2f}x")

Swamp WRN28-2 average inference time: 0.0148 seconds per batch (128 samples)
Swamp WRN28-2 average inference time per sample: 0.000116 seconds
IMP WRN28-2 average inference time: 0.0150 seconds per batch (128 samples)
IMP WRN28-2 average inference time per sample: 0.000117 seconds
Swamp модель быстрее на 1.01x


In [ ]:
import torch
import time

# Создаем тестовый входной тензор (предполагая, что вход - 3 канала, 32x32 как в CIFAR)
batch_size = 128
dummy_input = torch.randn(batch_size, 3, 32, 32).to(device)

# Функция для измерения времени inference
def measure_inference_time(model, input_tensor, num_runs=10000):
    # warm-up
    for _ in range(10):
        _ = model(input_tensor)

    # Синхронизация перед началом измерений
    torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(num_runs):
        _ = model(input_tensor)
    torch.cuda.synchronize()
    end_time = time.time()

    avg_time = (end_time - start_time) / num_runs
    return avg_time

# Измеряем время для swamp модели
model_test_swamp_wrn.eval()  # Переводим в режим оценки
with torch.no_grad():  # Отключаем вычисление градиентов
    swamp_time = measure_inference_time(model_test_swamp_wrn, dummy_input)
    print(f"Swamp WRN28-2 average inference time: {swamp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"Swamp WRN28-2 average inference time per sample: {swamp_time/batch_size:.6f} seconds")

# Измеряем время для imp модели
model_test_imp_wrn.eval()
with torch.no_grad():
    imp_time = measure_inference_time(model_test_imp_wrn, dummy_input)
    print(f"IMP WRN28-2 average inference time: {imp_time:.4f} seconds per batch ({batch_size} samples)")
    print(f"IMP WRN28-2 average inference time per sample: {imp_time/batch_size:.6f} seconds")

# Сравнение
speedup = imp_time / swamp_time if imp_time > swamp_time else swamp_time / imp_time
if imp_time < swamp_time:
    print(f"IMP модель быстрее на {speedup:.2f}x")
else:
    print(f"Swamp модель быстрее на {speedup:.2f}x")

Swamp WRN28-2 average inference time: 0.0148 seconds per batch (128 samples)
Swamp WRN28-2 average inference time per sample: 0.000116 seconds
IMP WRN28-2 average inference time: 0.0150 seconds per batch (128 samples)
IMP WRN28-2 average inference time per sample: 0.000117 seconds
Swamp модель быстрее на 1.01x


# Accuracy

In [26]:
model_test_imp1_wrn = WideResNet(depth=28, widen_factor=2).to(device)
model_test_imp1_wrn.load_state_dict(torch.load("imp1_wrn28_2_s4.pth"))

model_test_swamp_wrn = WideResNet(depth=28, widen_factor=2).to(device)
model_test_swamp_wrn.load_state_dict(torch.load("swamp_wrn28_2_s7.pth"))

<All keys matched successfully>

In [27]:
import torch

def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

acc_swamp = evaluate_model(model_test_swamp_wrn, testloader, device)
acc_imp = evaluate_model(model_test_imp1_wrn, testloader, device)

print(f"SWAMP Accuracy: {acc_swamp:.2f}%")
print(f"IMP Accuracy:   {acc_imp:.2f}%")
print(f"Δ Accuracy (SWAMP - IMP): {acc_swamp - acc_imp:.2f} p.p.")


SWAMP Accuracy: 88.38%
IMP Accuracy:   86.58%
Δ Accuracy (SWAMP - IMP): 1.80 p.p.
